# NBU rates ingestion
Set the default lakehouse to lh_bronze. Requires network access from Fabric runtime. Historical range must cover all orders.

In [ ]:
import requests
from datetime import date
from pyspark.sql import functions as F, types as T
from decimal import Decimal
start_date = '20230101'  # REPLACE: first order date
end_date = date.today().strftime('%Y%m%d')
records=[]
for code in ('usd','pln'):
    r=requests.get('https://bank.gov.ua/NBU_Exchange/exchange_site',params={'start':start_date,'end':end_date,'valcode':code,'json':''},timeout=60)
    r.raise_for_status()
    for x in r.json():
        records.append((code.upper(),str(x.get('exchangedate','')),str(x.get('rate','')),str(x)))
schema='currency STRING, raw_date STRING, raw_rate STRING, raw_payload STRING'
raw=spark.createDataFrame(records,schema).withColumn('ingested_at',F.current_timestamp())
raw.write.format('delta').mode('overwrite').saveAsTable('nbu_rates_raw')
display(raw.limit(10))
